In [1]:
from panricci.utils import GFALoader

In [29]:
# path_gfa = "/home/avila/UNIMIB/panricci/vg/20-sars-cov-2-ena.gfa"
path_gfa = "../data/example_same2_bubbles.gfa"
loader = GFALoader()

In [30]:
nodes, edges = loader.load_nodes_edges(path_gfa)
len(nodes), len(edges)

(18, 23)

In [31]:
nd = loader.node_depth(path_gfa, nodes)

In [32]:
nd["1"]

1.0

In [33]:
nd["2"]

0.5

In [34]:
nd["7"]

0.25

In [35]:
from panricci.node_distributions.variation_graph import DistributionNodes

In [36]:
G = loader(path_gfa)
distribution = DistributionNodes(G.to_undirected(), alpha=0.5)

In [37]:
# distribution._compute_distribution("3", ),distribution._compute_distribution("5", ),

In [38]:
G.is_directed()

True

In [39]:
distribution._compute_distribution("9", )

{'9': 0.5, '6': 0.125, '7': 0.0625, '8': 0.0625, '10': 0.25}

In [40]:
distribution._compute_distribution("5",)

{'5': 0.5, '4': 0.25, '6': 0.125, '7': 0.0625, '8': 0.0625}

In [41]:
from panricci.ricci_flow import RicciFlow


In [42]:
# run Ricci-Flow
ricci_flow = RicciFlow(G, 
                distribution,
                 tol_curvature=1e-15,
                log_level="INFO"
                )

<class 'networkx.utils.decorators.argmap'> compilation 4:3: FutureWarning: 

shortest_path will return an iterator that yields
(node, path) pairs instead of a dictionary when source
and target are unspecified beginning in version 3.5

To keep the current behavior, use:

	dict(nx.shortest_path(G))


In [43]:
ricci_graph = ricci_flow.run(
    iterations=100,
)

/home/avila/miniforge3/envs/panricci/lib/python3.13/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

[Ricci-Flow] 2025-04-13@15:08:00.635 | Stopping Ricci-Flow in iteration 26, all curvatures below tol=1e-15


In [44]:
print(ricci_graph)
weight_by_edge=dict()
for edge in ricci_graph.edges(data=True):
    print(edge)
    weight_by_edge[(edge[0], edge[1])] = edge[2]["weight"]

DiGraph with 18 nodes and 23 edges
('1', '2', {'weight': 0.25, 'curvature': 0.0})
('1', '3', {'weight': 0.25, 'curvature': 0.0})
('2', '4', {'weight': 0.3425925927094413, 'curvature': -2.220446049250313e-16})
('3', '4', {'weight': 0.3425925927094413, 'curvature': -2.220446049250313e-16})
('4', '5', {'weight': 0.3324074064387307, 'curvature': 5.551115123125783e-16})
('5', '6', {'weight': 0.33342697605906824, 'curvature': -6.661338147750939e-16})
('5', '7', {'weight': 0.33342697605906824, 'curvature': -6.661338147750939e-16})
('5', '8', {'weight': 0.33342697605906824, 'curvature': -6.661338147750939e-16})
('6', '9', {'weight': 0.3333237789552584, 'curvature': 8.881784197001252e-16})
('7', '9', {'weight': 0.3333237789552584, 'curvature': 8.881784197001252e-16})
('8', '9', {'weight': 0.3333237789552584, 'curvature': 8.881784197001252e-16})
('9', '10', {'weight': 0.33333524424838445, 'curvature': -4.440892098500626e-16})
('10', '11', {'weight': 0.33332377895525833, 'curvature': 6.6613381477

In [45]:
ricci_flow.wasserstein(distribution._compute_distribution("8"), distribution._compute_distribution("10"))

0.4999828012715629

In [46]:
from panricci.alignment.node_embeddings import NodeEmbeddings
node_embeddings = NodeEmbeddings(ricci_embedding=True, seq_embedding=False)

In [47]:
embeddings = node_embeddings(ricci_graph.to_directed())

[Ricci-Flow] 2025-04-13@15:08:02.178 | end - compute_node_embeddings


In [48]:
embeddings["6"],embeddings["7"],embeddings["8"]

(array([1.25842698, 2.25840978]),
 array([1.25842698, 2.25840978]),
 array([1.25842698, 2.25840978]))

In [49]:
embeddings["9"],embeddings["10"]

(array([1.59175075, 1.925086  ]), array([1.925086  , 1.59175075]))

In [50]:
embeddings["7"], embeddings["13"]

(array([1.25842698, 2.25840978]), array([2.59183675, 0.925     ]))

In [51]:
for node, emb in embeddings.items():
    print(node, emb)


1 [0.         3.51683675]
2 [0.25       3.26683675]
3 [0.25       3.26683675]
4 [0.59259259 2.92424416]
5 [0.925      2.59183675]
6 [1.25842698 2.25840978]
7 [1.25842698 2.25840978]
8 [1.25842698 2.25840978]
9 [1.59175075 1.925086  ]
10 [1.925086   1.59175075]
11 [2.25840978 1.25842698]
12 [2.25840978 1.25842698]
13 [2.59183675 0.925     ]
14 [2.92424416 0.59259259]
15 [3.26683675 0.25      ]
16 [3.26683675 0.25      ]
17 [3.26683675 0.25      ]
18 [3.51683675 0.        ]


In [52]:
from panricci.alignment.node_embeddings import shortest_paths

from_source, to_sink = shortest_paths(G)
to_sink["source"][1:-1]

['1', '2', '4', '5', '6', '9', '10', '11', '13', '14', '15', '18']

In [53]:
bubble_edges = [("1","2"),("1","3"),("3","4"),("2","4")] 
# sum([emb for  nodeid, emb in embeddings.items() if nodeid in bubble_nodes])

sum(weight_by_edge[edge] for edge in bubble_edges)

1.1851851854188826

In [54]:
bubble_edges = [("10","11"),("10","12"),("11","13"),("12","13")]

# sum([emb for  nodeid, emb in embeddings.items() if nodeid in bubble_nodes])
sum(weight_by_edge[edge] for edge in bubble_edges)

1.3335015100286531

In [55]:
bubble_edges = [
    ("5","6"),
    ("5","7"),
    ("5","8"),
    ("6","9"),
    ("7","9"),
    ("8","9"),
    ]

sum(weight_by_edge[edge] for edge in bubble_edges)

2.0002522650429797

In [56]:
bubble_edges = [
    ("14","15"),
    ("14","16"),
    ("14","17"),
    ("15","18"),
    ("16","18"),
    ("17","18"),
    ]

sum(weight_by_edge[edge] for edge in bubble_edges)

1.777777778128324